# Poincare-Einstein Metrics and Holographic Data

This notebook demonstrates the `conformal_toolkit.poincare_einstein` module,
which implements the Fefferman--Graham expansion and associated holographic
quantities central to the AdS/CFT correspondence.

## Mathematical background

A **Poincare-Einstein (PE) metric** is a conformally compact Einstein metric
on a manifold-with-boundary. Near the boundary (at $\rho = 0$), such a metric
takes the **Fefferman--Graham form**:
$$g_{\mathrm{bulk}} = \rho^{-2}\bigl(d\rho^2 + g_\rho\bigr)$$
where the **boundary metric expansion** is
$$g_\rho = g_0 + \rho^2\, g_2 + \rho^4\, g_4 + \cdots$$

The key result (Fefferman--Graham, 1985; Graham--Zworski, 2003) is that the
expansion coefficients $g_{2k}$ are **locally determined** by the boundary
conformal structure $[g_0]$ up to order $k < n/2$ (where $n = \dim \partial M$).
The first non-trivial coefficient is:
$$g_2 = -P(g_0)$$
where $P$ is the Schouten tensor of the boundary metric.

This determines several holographic quantities:
- **Dirichlet-to-Neumann operator**: maps boundary data to its normal derivative
- **Holographic stress tensor**: $T_{ab} \propto g_n$ (the first undetermined coefficient)
- **Holographic Weyl anomaly**: the conformal anomaly of the boundary CFT partition function
- **Renormalized volume**: the finite part of the bulk volume after holographic renormalization

In this notebook we start from simple boundary metrics and compute all of these.

## 1. FG expansion from the round $S^2$ boundary

We take the round 2-sphere as the boundary metric $g_0$. The 3D bulk is
hyperbolic space $\mathbb{H}^3$ (equivalently, Euclidean AdS$_3$).

The Schouten tensor of the round $S^2$ is $P = \frac{1}{2}g_0$ (using the 2D
convention), so $g_2 = -P = -\frac{1}{2}g_0$.

In [ ]:
from sage.all import Manifold, sin, cos, var
from conformal_toolkit.core.conformal_structure import ConformalStructure
from conformal_toolkit.poincare_einstein.fefferman_graham import (
    fg_coefficient_g2,
    fg_expansion,
)
from conformal_toolkit.poincare_einstein.dirichlet_neumann import (
    dirichlet_to_neumann,
    holographic_stress_tensor,
)
from conformal_toolkit.poincare_einstein.holographic_anomaly import (
    holographic_weyl_anomaly,
)
from conformal_toolkit.poincare_einstein.renormalized_volume import (
    renormalized_volume_coefficient,
)

# Round S^2 as boundary metric
S = Manifold(2, 'S2', structure='Riemannian')
X = S.chart(r'theta:(0,pi):\theta phi:(0,2*pi):\phi')
theta, phi = X[:]
g0 = S.metric('g0')
g0[0, 0] = 1
g0[1, 1] = sin(theta)**2

print("Boundary metric g_0 (round S^2):")
g0.display()

In [ ]:
# Compute g_2 = -P(g_0)
g2 = fg_coefficient_g2(g0)
print("FG coefficient g_2 = -P(g_0):")
g2.display()

# On S^2: P = (R/4)*g = (2/4)*g = g/2
# So g_2 = -g/2
#
# Output:
# g_2 = -1/2 dtheta*dtheta + (-1/2)*sin(theta)^2 dphi*dphi

In [ ]:
# Full FG expansion to order 2
coeffs = fg_expansion(g0, order=2)
print("FG expansion coefficients:")
for k, gk in sorted(coeffs.items()):
    print(f"\n  g_{k}:")
    gk.display()

# Output:
# g_0: dtheta*dtheta + sin(theta)^2 dphi*dphi
# g_2: -1/2 dtheta*dtheta + (-1/2)*sin(theta)^2 dphi*dphi
#
# The bulk metric near rho=0 is:
#   g_bulk = rho^{-2} [drho^2 + g_0 + rho^2 g_2 + ...]
#          = rho^{-2} [drho^2 + (1 - rho^2/2)(dtheta^2 + sin^2 theta dphi^2) + ...]

## 2. Dirichlet-to-Neumann operator

The conformal Dirichlet-to-Neumann (DN) operator maps boundary data to its
normal derivative at the conformal boundary. At leading order:
$$\mathrm{DN}(g_0) = g_2 = -P(g_0)$$

This is the boundary operator that encodes the response of the bulk to
boundary perturbations, and is central to the holographic dictionary.

In [ ]:
# Dirichlet-to-Neumann operator
DN = dirichlet_to_neumann(g0)
print("Dirichlet-to-Neumann operator DN(g_0):")
DN.display()

# Verify it equals g_2
print("\nDN(g_0) - g_2 (should vanish):")
diff = DN - g2
diff.display()

# Output:
# DN(g_0) = -1/2 dtheta*dtheta + (-1/2)*sin(theta)^2 dphi*dphi
# DN(g_0) - g_2: 0

## 3. Holographic Weyl anomaly

The holographic Weyl anomaly is the local density whose integral gives the
conformal anomaly of the boundary CFT. For a 2D boundary:
$$\mathcal{A} = \frac{R}{2}$$
where $R$ is the scalar curvature of $g_0$. This is the famous central charge
anomaly of 2D CFTs (related to the Polyakov action).

On the round $S^2$ with $R = 2$, the anomaly density is $\mathcal{A} = 1$.

In [ ]:
# Holographic Weyl anomaly for 2D boundary
anomaly = holographic_weyl_anomaly(g0)
print("Holographic Weyl anomaly density on S^2 boundary:")
anomaly.display()

# Output:
# anomaly = 1  (R/2 = 2/2 = 1 on the round S^2)

## 4. Renormalized volume coefficient

The renormalized volume of the PE metric is the finite part of the bulk
volume after subtracting divergences. The first non-trivial coefficient is:
$$v_2 = -\frac{1}{n-2}\,J = -\frac{R}{2(n-1)(n-2)}$$
In 2D, the formula is regularized: $v_2 = J/2$.

In [ ]:
# Renormalized volume coefficients
v0 = renormalized_volume_coefficient(g0, order=0)
v2 = renormalized_volume_coefficient(g0, order=2)

print("Renormalized volume coefficient v_0:")
print(f"  v_0 = {v0}")

print("\nRenormalized volume coefficient v_2:")
v2.display()

# On S^2: J = R/(2*(n-1)) = 2/(2*1) = 1
# 2D regularized formula: v_2 = J/2 = 1/2
#
# Output:
# v_0 = 1
# v_2 = 1/2

## 5. Holographic stress tensor

The holographic stress tensor is the boundary energy-momentum tensor dual to
the bulk gravity. At leading order:
$$T_{ab} = -n\,P_{ab}$$
In 3D (AdS$_4$/CFT$_3$) there is a trace correction:
$T_{ab} = -3P_{ab} + \frac{3}{2}Jg_{ab}$.

In [ ]:
# Holographic stress tensor for S^2 boundary (n=2)
T = holographic_stress_tensor(g0, n=2)
print("Holographic stress tensor T_{ab} on S^2 (n=2):")
T.display()

# T = -2 * P = -2 * (g/2) = -g
# Output:
# T = -dtheta*dtheta + (-sin(theta)^2) dphi*dphi

## 6. Flat $\mathbb{R}^3$ boundary: everything vanishes

As a sanity check, we take flat $\mathbb{R}^3$ as the boundary. The Schouten
tensor vanishes, so all FG coefficients, the DN operator, the anomaly, and
the renormalized volume coefficient should be zero (or trivial).

In [ ]:
# Flat R^3 as boundary
R3 = Manifold(3, 'R3', structure='Riemannian')
Z = R3.chart('x0 x1 x2')
g_flat = R3.metric('g_flat')
for i in range(3):
    g_flat[i, i] = 1

# g_2 on flat R^3: should be 0
g2_flat = fg_coefficient_g2(g_flat)
print("g_2 for flat R^3 boundary:")
g2_flat.display()

# Renormalized volume v_2: should vanish (J = 0)
v2_flat = renormalized_volume_coefficient(g_flat, order=2)
print("\nv_2 for flat R^3 boundary:")
v2_flat.display()

# Stress tensor: should vanish (P = 0)
T_flat = holographic_stress_tensor(g_flat, n=3)
print("\nHolographic stress tensor for flat R^3 boundary:")
T_flat.display()

# Output:
# g_2: 0
# v_2: 0
# T_{ab}: 0
#
# Everything vanishes, as expected for a flat boundary (bulk = Poincare half-space).

## 7. FG expansion to order 4 on $S^4$

For a 4D boundary, the FG expansion can be carried to order 4. The $g_4$
coefficient involves the square of $g_2$ and is only partially determined
when $n = 4$ (a logarithmic term appears). We compute the algebraically
determined piece.

In [ ]:
# Round S^4 as boundary metric
S4 = Manifold(4, 'S4', structure='Riemannian')
X4 = S4.chart(r'th1:(0,pi):\theta_1 th2:(0,pi):\theta_2 th3:(0,pi):\theta_3 ph:(0,2*pi):\varphi')
th1, th2, th3, ph = X4[:]
g0_s4 = S4.metric('g0')
g0_s4[0, 0] = 1
g0_s4[1, 1] = sin(th1)**2
g0_s4[2, 2] = sin(th1)**2 * sin(th2)**2
g0_s4[3, 3] = sin(th1)**2 * sin(th2)**2 * sin(th3)**2

# FG expansion to order 4
coeffs_s4 = fg_expansion(g0_s4, order=4)
print("FG expansion on S^4 boundary:")
for k, gk in sorted(coeffs_s4.items()):
    print(f"\n  g_{k}:")
    gk.display()

# Output:
# g_0: the round S^4 metric
# g_2: -P(g_0) = -(1/2)*Ric + (R/12)*g = -g/2 (since Ric = 3g, R = 12)
# g_4: (1/8)*tr(g_2^2)*g_0  (n=4 case, algebraically determined piece)

In [ ]:
# Holographic Weyl anomaly for 4D boundary (type-A anomaly = Q_4)
anomaly_s4 = holographic_weyl_anomaly(g0_s4)
print("Holographic Weyl anomaly on S^4 boundary (= Q_4):")
print(anomaly_s4.display())

# On the round S^4, Q_4 is a constant related to the curvature.
# The integrated anomaly (proportional to the Euler characteristic chi(S^4) = 2)
# gives the central charge of the dual CFT.

## Summary

We demonstrated the Poincare-Einstein module on two boundary metrics:

| Quantity | $S^2$ boundary | Flat $\mathbb{R}^3$ boundary |
|----------|---------------|-----------------------------|
| $g_2 = -P(g_0)$ | $-\frac{1}{2}g_0$ | $0$ |
| DN operator | $-\frac{1}{2}g_0$ | $0$ |
| Weyl anomaly | $1$ ($= R/2$) | N/A (odd $n$) |
| $v_2$ | $\frac{1}{2}$ | $0$ |
| Stress tensor | $-g_0$ | $0$ |

The FG expansion coefficients encode the holographic dictionary between
bulk gravity and boundary CFT. The `conformal_toolkit.poincare_einstein`
module makes these computations accessible within the SageMath ecosystem.

### Physical significance

- In AdS$_3$/CFT$_2$, the Weyl anomaly $R/2$ is proportional to the
  central charge $c$ of the boundary Virasoro algebra
- The FG expansion determines the asymptotic form of black hole metrics
  near the AdS boundary, connecting to holographic entanglement entropy
  and the Ryu--Takayanagi formula
- The renormalized volume appears in the computation of holographic
  complexity and the volume conjecture

**Next:** Notebook 05 bridges the symbolic computations of Track A with
the discrete ML features of Track B.